# Fair Comparison — Shared Patient Folds

**Purpose:** Generate a single, authoritative `patient_id -> fold` mapping that all three models
(VIPEEGNet, TCS-Net, EfficientNetB0) use for 5-fold cross-validation.

**Why this matters:** For a fair comparison, every model must validate on the *same set of patients*
in each fold. `sklearn.GroupKFold` is deterministic (it sorts unique groups internally), so calling
it with the same patient IDs yields the same folds everywhere. This notebook pins that down
explicitly by saving a CSV, so you can audit the splits and confirm consistency.

**Run this ONCE before running TCS-Net or EfficientNetB0.** VIPEEGNet doesn't need to be
re-run — just re-export its per-fold OOF predictions using the cell at the bottom of this notebook.

**Output:** `/workspace/fair_comparison/patient_folds.csv` with columns `[patient_id, fold]`.


In [2]:
!pip install numpy pandas scikit-learn -q

In [4]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

# --- Paths (edit for your environment) ---
DATA_DIR   = '/workspace/hms-data'
FAIR_DIR   = '/workspace/fair_comparison'
Path(FAIR_DIR).mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
SEED    = 2024  # documented for audit; GroupKFold itself is not stochastic

print(f'DATA_DIR  = {DATA_DIR}')
print(f'FAIR_DIR  = {FAIR_DIR}')
print(f'N_FOLDS   = {N_FOLDS}')

DATA_DIR  = /workspace/hms-data
FAIR_DIR  = /workspace/fair_comparison
N_FOLDS   = 5


In [5]:
# Load the full training manifest
df_full = pd.read_csv(f'{DATA_DIR}/train.csv')
print(f'Full train.csv: {len(df_full):,} rows, {df_full["patient_id"].nunique():,} patients, '
      f'{df_full["eeg_id"].nunique():,} eeg_ids')

Full train.csv: 106,800 rows, 1,950 patients, 17,089 eeg_ids


In [6]:
# Build deterministic patient -> fold mapping.
# GroupKFold internally calls np.unique(groups), which sorts, so this is stable
# regardless of the order rows appear in the input dataframe.

# Use one row per patient (we only need the patient_id column for the split)
patients = df_full[['patient_id']].drop_duplicates().sort_values('patient_id').reset_index(drop=True)
print(f'Unique patients: {len(patients):,}')

gkf = GroupKFold(n_splits=N_FOLDS)
patients['fold'] = -1
for k, (_, vi) in enumerate(gkf.split(patients, groups=patients['patient_id'].values)):
    patients.loc[vi, 'fold'] = k

# Sanity: every patient assigned, each fold roughly balanced
print('\nFold distribution (patients):')
print(patients.groupby('fold').size().rename('n_patients').to_frame())
assert (patients['fold'] >= 0).all(), 'some patients not assigned'

Unique patients: 1,950

Fold distribution (patients):
      n_patients
fold            
0            390
1            390
2            390
3            390
4            390


In [7]:
# Join fold back to the full manifest to see segment-level counts per fold
df_check = df_full.merge(patients, on='patient_id', how='left')

# HQ subset (votes >= 10) at the segment level -- matches VIPEEGNet/TCS-Net eval protocol
LABEL_COLS = ['seizure_vote','lpd_vote','gpd_vote','lrda_vote','grda_vote','other_vote']
df_check['total_votes'] = df_check[LABEL_COLS].sum(1)

print(f'{"Fold":>4} | {"Patients":>9} | {"Segments":>9} | {"HQ segs":>9} | {"eeg_ids":>8}')
print('-' * 60)
for k in range(N_FOLDS):
    fd = df_check[df_check.fold == k]
    hq = fd[fd.total_votes >= 10]
    print(f'{k:>4} | {fd["patient_id"].nunique():>9,} | {len(fd):>9,} | '
          f'{len(hq):>9,} | {fd["eeg_id"].nunique():>8,}')

Fold |  Patients |  Segments |   HQ segs |  eeg_ids
------------------------------------------------------------
   0 |       390 |    19,384 |     5,721 |    3,091
   1 |       390 |    21,618 |     8,450 |    3,066
   2 |       390 |    19,307 |     6,642 |    3,809
   3 |       390 |    21,544 |     7,992 |    3,497
   4 |       390 |    24,947 |    11,141 |    3,626


/tmp/ipykernel_1742/4004809687.py:6: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  df_check['total_votes'] = df_check[LABEL_COLS].sum(1)


In [8]:
# Save the shared mapping
out = f'{FAIR_DIR}/patient_folds.csv'
patients.to_csv(out, index=False)
print(f'\nSaved {out}')
print(f'  rows: {len(patients):,}')
print(f'  cols: {list(patients.columns)}')

# Also save a manifest with metadata
import json, datetime
with open(f'{FAIR_DIR}/splits_manifest.json', 'w') as f:
    json.dump({
        'created_utc': datetime.datetime.utcnow().isoformat() + 'Z',
        'n_folds': N_FOLDS,
        'seed_documented': SEED,
        'method': 'sklearn.GroupKFold on sorted unique patient_ids',
        'n_patients': int(len(patients)),
        'n_eeg_ids': int(df_full['eeg_id'].nunique()),
        'n_segments': int(len(df_full)),
        'per_fold_patients': {int(k): int(v) for k, v in patients.groupby('fold').size().items()},
    }, f, indent=2)
print(f'Saved {FAIR_DIR}/splits_manifest.json')


Saved /workspace/fair_comparison/patient_folds.csv
  rows: 1,950
  cols: ['patient_id', 'fold']
Saved /workspace/fair_comparison/splits_manifest.json


/tmp/ipykernel_1742/2040547490.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_utc': datetime.datetime.utcnow().isoformat() + 'Z',


## Consistency check

Confirm that each model's native dataframe, when merged with `patient_folds.csv`, reproduces the
fold structure its native split would have produced. If these match, you can trust the folds.


In [10]:
# What actually matters for fairness: every row's patient_id gets a fold assignment,
# and no patient ever appears in two different folds.

def check_df(df_name, df):
    merged = df.merge(patients, on='patient_id', how='left')
    assert merged['fold'].notna().all(), f'{df_name}: some rows have no fold — patient_id missing'
    # No patient should straddle folds
    per_patient = merged.groupby('patient_id')['fold'].nunique()
    assert (per_patient == 1).all(), f'{df_name}: some patient appears in multiple folds'
    print(f'{df_name}:')
    print(f'  rows: {len(merged):,}  patients: {merged["patient_id"].nunique():,}')
    print(f'  fold sizes: {dict(sorted(merged.groupby("fold").size().items()))}')
    return merged

# Check 1: VIPEEGNet / TCS-Net deduplication
dedup = df_full.drop_duplicates(['eeg_id'] + LABEL_COLS).reset_index(drop=True)
check_df('dedup df (VIPEEGNet / TCS-Net)', dedup)

# Check 2: EfficientNetB0 eeg_id aggregation
agg = df_full.groupby('eeg_id')[['patient_id']].first().reset_index()
check_df('eeg_id-agg df (EfficientNetB0)', agg)

print('\n✓ Every row in both native dataframes gets a fold via patient_id merge.')
print('  No patient crosses folds. Fair comparison is guaranteed.')

dedup df (VIPEEGNet / TCS-Net):
  rows: 20,183  patients: 1,950
  fold sizes: {0: 3831, 1: 3655, 2: 4483, 3: 4186, 4: 4028}
eeg_id-agg df (EfficientNetB0):
  rows: 17,089  patients: 1,950
  fold sizes: {0: 3091, 1: 3066, 2: 3809, 3: 3497, 4: 3626}

✓ Every row in both native dataframes gets a fold via patient_id merge.
  No patient crosses folds. Fair comparison is guaranteed.


## (Optional) Re-export VIPEEGNet OOF predictions in the shared format

If your VIPEEGNet run already finished, run this cell once to convert its outputs to the shared
`.npz` + `summary.json` format that the comparison notebook will load. You don't need to re-train.

Skip this cell if VIPEEGNet hasn't been trained yet — just point the fair-VIPEEGNet output there.


In [ ]:
# Re-export VIPEEGNet outputs (edit path to wherever your VIPEEGNet saved weights/preds)
VIPEEG_OUT = '/workspace/vipeegnet/output'
FAIR_VIPEEG = f'{FAIR_DIR}/vipeegnet'
Path(FAIR_VIPEEG).mkdir(parents=True, exist_ok=True)

# This cell is a template — adapt to how your VIPEEGNet saved its predictions.
# Assumption: you saved per-fold predictions as you evaluated them in the original notebook.
# If you only saved the aggregated all_yt/all_yp arrays, you'll need to split by fold_boundaries.

print(f'Re-export stub. Adapt to your VIPEEGNet output structure.')
print(f'Expected final format for each fold:')
print(f'  {FAIR_VIPEEG}/fold{{k}}_oof.npz with keys: yt, yp, wt (optional), eeg_ids (optional)')
print(f'  {FAIR_VIPEEG}/summary.json with: model, per_fold_kld, oof_kld, config')